In [1]:
#!/usr/bin/env python3
import os
import shutil
import subprocess
import logging
from pathlib import Path
from typing import Optional, Dict, List, Set

THREADS = 64
QUAL_THRESHOLD = 20
MIN_READ_LEN = 50
LONG_QUAL_THRESHOLD = 15
LONG_MIN_LEN = 500
BUSCO_DB = "bacteria_odb12.2"
BUSCO_DB_DOWNLOAD = "/active-data/datasets/busco_downloads"
BASE_WORK_ROOT = Path("/active-data/genome_re-assemble")

class ToolRunError(Exception):
    pass

def init_logger(log_path: Path):
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_path, encoding="utf-8"),
            logging.StreamHandler()
        ],
        force=True
    )
    return logging.getLogger(__name__)

def make_dirs(dir_list: List[Path]):
    for d in dir_list:
        d.mkdir(parents=True, exist_ok=True)
        if not os.access(d, os.W_OK):
            raise PermissionError(f"No write permission for directory: {d}")

def run_cmd(logger: logging.Logger, cmd: list, desc: str):
    cmd_str = ' '.join(cmd)
    logger.info(f"[{desc}] Execute command: {cmd_str}")
    try:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        stdout, stderr = proc.communicate()
        if proc.returncode != 0:
            err_msg = f"[{desc}] Execution failed, return code: {proc.returncode}\nSTDERR:\n{stderr}"
            logger.error(err_msg)
            raise ToolRunError(err_msg)
    except ToolRunError:
        raise
    except Exception as e:
        err_msg = f"[{desc}] Unknown exception during execution: {str(e)}"
        logger.error(err_msg)
        raise ToolRunError(err_msg)
    logger.info(f"[{desc}] Execution completed")

def validate_sra(logger: logging.Logger, sra_path: Path) -> bool:
    if not sra_path.exists():
        logger.warning(f"SRA file not found: {sra_path}")
        return False
    cmd = ["vdb-validate", str(sra_path)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    out, err = proc.communicate()
    if proc.returncode == 0:
        logger.info(f"{sra_path.name} SRA validation passed")
        return True
    else:
        logger.error(f"{sra_path.name} vdb-validate failed, file corrupted")
        return False

def download_single_srr(logger: logging.Logger, srr_id: str, raw_sra_dir: Path):
    sra_sub_dir = raw_sra_dir / srr_id
    sra_file = sra_sub_dir / f"{srr_id}.sra"
    if sra_file.exists():
        try:
            ok = validate_sra(logger, sra_file)
        except Exception:
            ok = False
        if ok:
            logger.info(f"{srr_id} SRA validated, skip download")
            return
        else:
            logger.warning(f"{srr_id} SRA corrupted, remove and re-download")
            shutil.rmtree(sra_sub_dir)
    cmd = [
        "prefetch",
        "--max-size", "200G",
        "-O", str(raw_sra_dir),
        srr_id
    ]
    try:
        run_cmd(logger, cmd, f"prefetch download {srr_id}")
    except ToolRunError as e:
        logger.error(f"{srr_id} download failed, skip this SRR: {e}")
        return
    if not validate_sra(logger, sra_file):
        logger.error(f"{srr_id} validation failed after download, discard this SRR")

def dump_single_fastq(logger: logging.Logger, srr_id: str, layout: str, raw_sra_dir: Path, raw_fastq_dir: Path) -> str:
    sra_path = raw_sra_dir / srr_id / f"{srr_id}.sra"
    if not sra_path.exists():
        logger.error(f"{srr_id} SRA not found: {sra_path}")
        raise ToolRunError(f"SRA file missing for {srr_id}")
    layout = layout.lower()
    if layout not in ("paired", "single"):
        logger.error(f"Invalid layout value: {layout}, only paired/single supported")
        raise ToolRunError(f"Illegal layout {layout}")

    fq_exists = False
    if layout == "paired":
        f1 = raw_fastq_dir / f"{srr_id}_1.fastq"
        f2 = raw_fastq_dir / f"{srr_id}_2.fastq"
        if f1.exists() and f2.exists() and f1.stat().st_size > 0 and f2.stat().st_size > 0:
            fq_exists = True
    else:
        f_single = raw_fastq_dir / f"{srr_id}.fastq"
        if f_single.exists() and f_single.stat().st_size > 0:
            fq_exists = True

    if fq_exists:
        logger.info(f"{srr_id} fastq already exists, skip fasterq-dump")
        return layout

    cmd = ["fasterq-dump", "-O", str(raw_fastq_dir), str(sra_path)]
    if layout == "paired":
        cmd.insert(1, "--split-files")
    run_cmd(logger, cmd, f"Export fastq for {srr_id}")
    return layout

def run_raw_fastqc(logger: logging.Logger, srr_id: str, raw_fastq_dir: Path, qc_root: Path):
    out_dir = qc_root / f"raw_fastqc_{srr_id}"
    html_report = out_dir / f"{srr_id}_1_fastqc.html"
    if html_report.exists() and html_report.stat().st_size > 0:
        logger.info(f"{srr_id} FastQC report exists, skip QC")
        return

    out_dir.mkdir(exist_ok=True)
    fq_list = list(raw_fastq_dir.glob(f"{srr_id}*.fastq"))
    if not fq_list:
        logger.warning(f"No raw fastq found for {srr_id}, skip FastQC")
        return
    try:
        cmd = ["fastqc", "-o", str(out_dir), "-t", str(THREADS)] + [str(f) for f in fq_list]
        run_cmd(logger, cmd, f"Raw FastQC for {srr_id}")
    except ToolRunError as e:
        logger.warning(f"FastQC failed for {srr_id}, continue workflow: {e}")

def clean_illumina_short(logger: logging.Logger, srr_id: str, raw_fastq_dir: Path, clean_short_dir: Path):
    r1_in = raw_fastq_dir / f"{srr_id}_1.fastq"
    r2_in = raw_fastq_dir / f"{srr_id}_2.fastq"
    r1_out = clean_short_dir / f"{srr_id}_clean_R1.fq.gz"
    r2_out = clean_short_dir / f"{srr_id}_clean_R2.fq.gz"

    if r1_out.exists() and r2_out.exists() and r1_out.stat().st_size > 0 and r2_out.stat().st_size > 0:
        logger.info(f"Cleaned fq.gz for {srr_id} exists, skip fastp short read cleaning")
        return

    cmd = [
        "fastp",
        "-i", str(r1_in), "-I", str(r2_in),
        "-o", str(r1_out), "-O", str(r2_out),
        "--qualified_quality_phred", str(QUAL_THRESHOLD),
        "--length_required", str(MIN_READ_LEN),
        "--n_base_limit", "5",
        "--thread", str(THREADS)
    ]
    try:
        run_cmd(logger, cmd, f"Clean Illumina short reads {srr_id}")
    except ToolRunError as e:
        logger.warning(f"Short read cleaning failed for {srr_id}, skip this library in assembly: {e}")

def clean_long_read(logger: logging.Logger, srr_id: str, raw_fastq_dir: Path, clean_long_dir: Path, platform: str):
    fq_in = raw_fastq_dir / f"{srr_id}.fastq"
    fq_out = clean_long_dir / f"{srr_id}_clean.fq.gz"

    if fq_out.exists() and fq_out.stat().st_size > 0:
        try:
            check_gz_cmd = ["gunzip", "-t", str(fq_out)]
            subprocess.run(check_gz_cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            logger.info(f"Cleaned long read file {srr_id} valid, skip filtering")
            return
        except subprocess.CalledProcessError:
            logger.warning(f"Cleaned file corrupted for {srr_id}, re-filter")
            fq_out.unlink()

    if fq_out.exists():
        fq_out.unlink()

    cmd = [
        "fastp",
        "-i", str(fq_in),
        "-o", str(fq_out),
        "--qualified_quality_phred", str(LONG_QUAL_THRESHOLD),
        "--length_required", str(LONG_MIN_LEN),
        "--thread", str(THREADS)
    ]
    if platform.upper() == "PACBIO_SMRT":
        cmd.append("--phred64")
    cmd.append("--disable_trim_poly_g")

    try:
        run_cmd(logger, cmd, f"First round long read filtering {platform} {srr_id}")
        check_gz_cmd = ["gunzip", "-t", str(fq_out)]
        subprocess.run(check_gz_cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        MIN_VALID_SIZE = 50 * 1024 * 1024
        file_size = fq_out.stat().st_size
        if file_size < MIN_VALID_SIZE:
            logger.warning(f"Cleaned file size too small for {srr_id}, disable quality filter and reprocess")
            fq_out.unlink()
            cmd.append("--disable_quality_filtering")
            run_cmd(logger, cmd, f"Second round long read filtering (quality filter off) {platform} {srr_id}")
            subprocess.run(check_gz_cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    except ToolRunError as e:
        logger.warning(f"Long read cleaning failed for {srr_id}, exclude from assembly: {e}")

def unicycler_hybrid_assemble(logger: logging.Logger, r1_list: List[Path], r2_list: List[Path], long_list: List[Path], assemble_dir: Path) -> Optional[Path]:
    final_genome = assemble_dir / "assembly.fasta"
    if final_genome.exists() and final_genome.stat().st_size > 0:
        logger.info(f"Unicycler assembly result exists, skip assembly")
        return final_genome

    if not r1_list or not r2_list:
        logger.error("Illumina short reads missing, hybrid assembly aborted")
        return None
    if not long_list:
        logger.error("Long read data missing")
        return None

    cmd = ["unicycler", "-t", str(THREADS), "--min_fasta_length", "1000", "-o", str(assemble_dir), "--mode", "conservative"]
    for r1, r2 in zip(r1_list, r2_list):
        cmd.extend(["-1", str(r1), "-2", str(r2)])
    for lfq in long_list:
        cmd.extend(["-l", str(lfq)])

    try:
        run_cmd(logger, cmd, "Unicycler hybrid assembly")
    except ToolRunError as e:
        logger.error(f"Unicycler assembly failed: {e}")
        return None
    if not final_genome.exists():
        logger.error("assembly.fasta not generated by Unicycler, assembly failed")
        return None
    logger.info(f"Unicycler assembly finished: {final_genome.name}")
    return final_genome

def spades_hybrid_assemble(
    logger: logging.Logger,
    r1_list: List[Path],
    r2_list: List[Path],
    long_read_runs: List[Dict],
    raw_fastq_dir: Path,
    clean_long_dir: Path,
    skip_clean: bool,
    assemble_dir: Path,
) -> Optional[Path]:
    final_genome = assemble_dir / "scaffolds.fasta"
    if final_genome.exists() and final_genome.stat().st_size > 0:
        logger.info(f"SPAdes assembly result exists, skip assembly")
        return final_genome

    if not r1_list or not r2_list:
        logger.error("Illumina short reads missing, SPAdes assembly aborted")
        return None

    cmd = [
        "spades.py",
        "-t", str(THREADS),
        "-o", str(assemble_dir)
    ]
    pe_idx = 1
    for r1, r2 in zip(r1_list, r2_list):
        cmd.extend([f"-{pe_idx}", str(r1), f"-{pe_idx+1}", str(r2)])
        pe_idx += 2

    for run in long_read_runs:
        srr = run["run_id"]
        plat = run["platform"].upper()
        if skip_clean:
            fq_path = raw_fastq_dir / f"{srr}.fastq"
        else:
            fq_path = clean_long_dir / f"{srr}_clean.fq.gz"
        if not fq_path.exists():
            logger.error(f"Long read file missing: {fq_path}")
            continue
        if plat == "OXFORD_NANOPORE":
            cmd.extend(["--nanopore", str(fq_path)])
        elif plat == "PACBIO_SMRT":
            cmd.extend(["--pacbio", str(fq_path)])
        else:
            logger.warning(f"Unknown sequencing platform {plat}, skip {srr}")

    try:
        run_cmd(logger, cmd, "SPAdes hybrid assembly")
    except ToolRunError as e:
        logger.error(f"SPAdes assembly failed: {e}")
        return None
    if not final_genome.exists():
        logger.error("scaffolds.fasta not generated by SPAdes, assembly failed")
        return None
    logger.info(f"SPAdes assembly finished: {final_genome.name}")
    return final_genome

def hybracter_hybrid_assemble(
    logger: logging.Logger,
    accession: str,
    r1_list: List[Path],
    r2_list: List[Path],
    long_read_runs: List[Dict],
    raw_fastq_dir: Path,
    clean_long_dir: Path,
    skip_clean: bool,
    assemble_dir: Path,
    est_genome_size: Optional[int]
) -> Optional[Path]:
    sample_name = accession
    path_complete = assemble_dir / "FINAL_OUTPUT" / "complete" / f"{sample_name}_final.fasta"
    path_incomplete = assemble_dir / "FINAL_OUTPUT" / "incomplete" / f"{sample_name}_final.fasta"

    final_fasta = None
    if path_complete.exists() and path_complete.stat().st_size > 0:
        final_fasta = path_complete
    elif path_incomplete.exists() and path_incomplete.stat().st_size > 0:
        final_fasta = path_incomplete

    if final_fasta is not None:
        logger.info(f"Hybracter assembly result exists, skip assembly")
        return final_fasta

    try:
        if not r1_list or not r2_list:
            raise ValueError("Hybracter requires Illumina short reads")
        if not long_read_runs:
            raise ValueError("Hybracter requires long read data")

        long_fq_all = []
        for run in long_read_runs:
            srr = run["run_id"]
            if skip_clean:
                fq = raw_fastq_dir / f"{srr}.fastq"
            else:
                fq = clean_long_dir / f"{srr}_clean.fq.gz"
            if not fq.exists():
                raise FileNotFoundError(f"Long read file missing for Hybracter: {fq}")
            long_fq_all.append(fq)

        if len(r1_list) > 1:
            logger.warning(f"Multiple short read sets detected, only use the first pair")
        use_r1 = r1_list[0]
        use_r2 = r2_list[0]

        if len(long_fq_all) > 1:
            logger.warning(f"Multiple long read files detected, only use the first file")
        use_long = long_fq_all[0]

        cmd = [
            "hybracter", "hybrid-single",
            "-s", sample_name,
            "-1", str(use_r1),
            "-2", str(use_r2),
            "-l", str(use_long),
            "-o", str(assemble_dir),
            "-t", str(THREADS),
            "--logic", "best"
        ]
        if est_genome_size is not None:
            cmd.extend(["-c", str(est_genome_size)])
        else:
            cmd.append("--auto")

        run_cmd(logger, cmd, "Hybracter fully automatic hybrid assembly")

        final_fasta = None
        if path_complete.exists() and path_complete.stat().st_size > 0:
            final_fasta = path_complete
            logger.info(f"Hybracter generated closed complete genome")
        elif path_incomplete.exists() and path_incomplete.stat().st_size > 0:
            final_fasta = path_incomplete
            logger.warning(f"Hybracter generated fragmented incomplete genome")

        if final_fasta is None:
            raise FileNotFoundError("No valid genome sequence output from Hybracter")

    except (ToolRunError, ValueError, FileNotFoundError, Exception) as e:
        logger.error(f"Hybracter assembly failed, skip this assembler: {str(e)}")
        return None
    return final_fasta

def flye_hybrid_assemble(
    logger: logging.Logger,
    long_read_runs: List[Dict],
    raw_fastq_dir: Path,
    clean_long_dir: Path,
    skip_clean: bool,
    assemble_dir: Path,
    est_genome_size: Optional[int]
) -> Optional[Path]:
    final_genome = assemble_dir / "assembly.fasta"
    if final_genome.exists() and final_genome.stat().st_size > 0:
        logger.info(f"Flye assembly result exists, skip assembly")
        return final_genome

    if not long_read_runs:
        logger.error("Flye assembly requires Nanopore/PacBio long reads")
        return None

    long_fq = None
    long_platform = None
    for run in long_read_runs:
        srr = run["run_id"]
        plat = run["platform"].upper()
        if skip_clean:
            fq_path = raw_fastq_dir / f"{srr}.fastq"
        else:
            fq_path = clean_long_dir / f"{srr}_clean.fq.gz"
        if not fq_path.exists():
            logger.warning(f"Skip missing long read for Flye: {fq_path}")
            continue
        long_fq = fq_path
        long_platform = plat
        break
    if long_fq is None:
        logger.error("No available long read files, Flye assembly terminated")
        return None

    cmd = [
        "flye",
        "-t", str(THREADS),
        "-o", str(assemble_dir)
    ]
    if long_platform == "OXFORD_NANOPORE":
        cmd.extend(["--nano-raw", str(long_fq)])
    elif long_platform == "PACBIO_SMRT":
        cmd.extend(["--pacbio-raw", str(long_fq)])
    else:
        logger.error(f"Unsupported sequencing platform for Flye {long_platform}")
        return None

    if est_genome_size:
        cmd.extend(["--genome-size", f"{est_genome_size}"])

    try:
        run_cmd(logger, cmd, "Flye long-read only assembly")
    except ToolRunError as e:
        logger.error(f"Flye assembly execution failed: {e}")
        return None

    if not final_genome.exists():
        logger.error("assembly.fasta not generated by Flye, assembly failed")
        return None
    logger.info(f"Flye assembly finished, output sequence: {final_genome.name}")
    return final_genome

def run_quast(logger: logging.Logger, genome_fa: Path, out_prefix: Path, est_genome_size: Optional[int] = None):
    cmd = [
        "quast.py",
        str(genome_fa),
        "-o", str(out_prefix),
        "-t", str(THREADS),
        "--circos"
    ]
    if est_genome_size is not None:
        cmd.extend(["--est-ref-size", str(est_genome_size)])
    try:
        run_cmd(logger, cmd, f"QUAST evaluation for {genome_fa.name}")
    except ToolRunError as e:
        logger.warning(f"QUAST quality control failed, skip: {e}")

def run_busco(logger: logging.Logger, genome_fa: Path, out_prefix: Path):
    cmd = [
        "busco",
        "-i", str(genome_fa),
        "-l", BUSCO_DB,
        "-o", str(out_prefix),
        "-m", "genome",
        "-c", str(THREADS),
        "--download_path", BUSCO_DB_DOWNLOAD,
        "-f"
    ]
    try:
        run_cmd(logger, cmd, f"BUSCO completeness check for {genome_fa.name}")
    except ToolRunError as e:
        logger.warning(f"BUSCO completeness assessment failed, skip: {e}")

def check_env(logger: logging.Logger, enable_assemblers: Set[str]):
    logger.info("Start validating dependency software versions")
    sra_tools = ["vdb-config", "vdb-validate", "prefetch", "fasterq-dump"]
    for tool in sra_tools:
        try:
            run_cmd(logger, [tool, "--version"], f"Validate {tool}")
        except ToolRunError:
            logger.error(f"Critical dependency {tool} missing, workflow cannot run")
            raise SystemExit(1)
    try:
        run_cmd(logger, ["fastqc", "--version"], "Validate fastqc")
        run_cmd(logger, ["fastp", "--version"], "Validate fastp")
        run_cmd(logger, ["quast.py", "--version"], "Validate quast")
        run_cmd(logger, ["busco", "--version"], "Validate busco")
    except ToolRunError as e:
        logger.error(f"Basic QC tool missing: {e}")
        raise SystemExit(1)

    if "unicycler" in enable_assemblers:
        run_cmd(logger, ["unicycler", "--version"], "Validate unicycler")
    if "spades" in enable_assemblers:
        run_cmd(logger, ["spades.py", "--version"], "Validate SPAdes")
    if "hybracter" in enable_assemblers:
        run_cmd(logger, ["hybracter", "version"], "Validate Hybracter")
    if "flye" in enable_assemblers:
        run_cmd(logger, ["flye", "--version"], "Validate Flye")

    logger.info("All dependency validation passed")

def hybrid_assemble_genome(
    accession: str,
    run_info_list: List[Dict],
    use_unicycler: bool = True,
    use_spades: bool = False,
    use_hybracter: bool = False,
    use_flye: bool = False,
    est_genome_size: Optional[int] = None,
    skip_clean: bool = False,
    skip_qc: bool = False,
) -> Dict[str, Optional[Path]]:
    if not any([use_unicycler, use_spades, use_hybracter, use_flye]):
        raise ValueError("At least one assembler must be enabled: unicycler/spades/hybracter/flye")

    suffix_parts = []
    if skip_clean:
        suffix_parts.append("skipClean")
    if skip_qc:
        suffix_parts.append("skipQC")
    dir_suffix = "_".join(suffix_parts) if suffix_parts else ""

    WORK_DIR = BASE_WORK_ROOT / accession
    DIR_RAW_SRA = WORK_DIR / "raw_sra"
    DIR_RAW_FASTQ = WORK_DIR / "raw_fastq"
    DIR_CLEAN_SHORT = WORK_DIR / "clean_fastq_short"
    DIR_CLEAN_LONG = WORK_DIR / "clean_fastq_long"

    base_uni = "assembly_unicycler"
    base_sp = "assembly_spades"
    base_hyb = "assembly_hybracter"
    base_flye = "assembly_flye"
    if dir_suffix:
        DIR_ASSEMBLY_UNI = WORK_DIR / f"{base_uni}_{dir_suffix}"
        DIR_ASSEMBLY_SP = WORK_DIR / f"{base_sp}_{dir_suffix}"
        DIR_ASSEMBLY_HYB = WORK_DIR / f"{base_hyb}_{dir_suffix}"
        DIR_ASSEMBLY_FLYE = WORK_DIR / f"{base_flye}_{dir_suffix}"
        DIR_QC = WORK_DIR / f"qc_report_{dir_suffix}"
    else:
        DIR_ASSEMBLY_UNI = WORK_DIR / base_uni
        DIR_ASSEMBLY_SP = WORK_DIR / base_sp
        DIR_ASSEMBLY_HYB = WORK_DIR / base_hyb
        DIR_ASSEMBLY_FLYE = WORK_DIR / base_flye
        DIR_QC = WORK_DIR / "qc_report"

    DIR_LOG = WORK_DIR / "logs"

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)

    all_dirs = [
        DIR_RAW_SRA, DIR_RAW_FASTQ, DIR_CLEAN_SHORT, DIR_CLEAN_LONG,
        DIR_ASSEMBLY_UNI, DIR_ASSEMBLY_SP, DIR_ASSEMBLY_HYB, DIR_ASSEMBLY_FLYE, DIR_QC, DIR_LOG
    ]
    make_dirs(all_dirs)

    log_file = DIR_LOG / f"{accession}_hybrid_assemble.log"
    logger = init_logger(log_file)

    logger.info(f"========== Hybrid assembly pipeline started for {accession} ==========")
    logger.info(f"Working directory: {WORK_DIR.resolve()}")
    logger.info(f"Assemblers enabled: Uni={use_unicycler}, SPAdes={use_spades}, Hyb={use_hybracter}, Flye={use_flye}")
    logger.info(f"skip_clean={skip_clean}, skip_qc={skip_qc}")

    enable_set = set()
    if use_unicycler:
        enable_set.add("unicycler")
    if use_spades:
        enable_set.add("spades")
    if use_hybracter:
        enable_set.add("hybracter")
    if use_flye:
        enable_set.add("flye")
    check_env(logger, enable_assemblers=enable_set)

    short_read_runs = []
    long_read_runs = []
    for run in run_info_list:
        rid = run["run_id"]
        plat = run["platform"].upper()
        if plat == "ILLUMINA":
            short_read_runs.append(run)
        elif plat in ("PACBIO_SMRT", "OXFORD_NANOPORE"):
            long_read_runs.append(run)
        else:
            logger.warning(f"Unsupported sequencing platform {plat}, skip {rid}")

    if not short_read_runs:
        logger.error("Illumina short reads missing, terminate sample workflow")
        return {
            "unicycler_fasta": None,
            "spades_fasta": None,
            "hybracter_fasta": None,
            "flye_fasta": None
        }
    if not long_read_runs:
        logger.error("Long read data missing, terminate sample workflow")
        return {
            "unicycler_fasta": None,
            "spades_fasta": None,
            "hybracter_fasta": None,
            "flye_fasta": None
        }

    all_srr_ids = [r["run_id"] for r in run_info_list]
    for srr in all_srr_ids:
        download_single_srr(logger, srr, DIR_RAW_SRA)
    for run in run_info_list:
        try:
            dump_single_fastq(logger, run["run_id"], run["layout"], DIR_RAW_SRA, DIR_RAW_FASTQ)
        except ToolRunError as e:
            logger.error(f"fastq export failed for {run['run_id']}: {e}")

    if not skip_qc:
        logger.info("Execute raw FastQC quality control")
        for srr in all_srr_ids:
            run_raw_fastqc(logger, srr, DIR_RAW_FASTQ, DIR_QC)
    else:
        logger.info("Skip FastQC quality control")

    if not skip_clean:
        logger.info("Execute fastp data cleaning")
        for run in short_read_runs:
            clean_illumina_short(logger, run["run_id"], DIR_RAW_FASTQ, DIR_CLEAN_SHORT)
        for run in long_read_runs:
            clean_long_read(logger, run["run_id"], DIR_RAW_FASTQ, DIR_CLEAN_LONG, run["platform"])
    else:
        logger.info("Skip fastp cleaning, use raw fastq for assembly")

    if skip_clean:
        r1_list = sorted(list(DIR_RAW_FASTQ.glob("*_1.fastq")))
        r2_list = sorted(list(DIR_RAW_FASTQ.glob("*_2.fastq")))
    else:
        r1_list = sorted(list(DIR_CLEAN_SHORT.glob("*_clean_R1.fq.gz")))
        r2_list = sorted(list(DIR_CLEAN_SHORT.glob("*_clean_R2.fq.gz")))

    if skip_clean:
        all_fq = list(DIR_RAW_FASTQ.glob("*.fastq"))
        long_list = sorted([fq for fq in all_fq if not fq.stem.endswith(("_1", "_2"))])
    else:
        long_list = sorted(list(DIR_CLEAN_LONG.glob("*_clean.fq.gz")))

    uni_fasta: Optional[Path] = None
    sp_fasta: Optional[Path] = None
    hyb_fasta: Optional[Path] = None
    flye_fasta: Optional[Path] = None

    if use_unicycler:
        logger.info("===== Start Unicycler Assembly =====")
        uni_fasta = unicycler_hybrid_assemble(logger, r1_list, r2_list, long_list, DIR_ASSEMBLY_UNI)
        if uni_fasta is not None:
            run_quast(logger, uni_fasta, DIR_QC / "quast_unicycler", est_genome_size)
            run_busco(logger, uni_fasta, DIR_QC / "busco_unicycler")
        else:
            logger.warning("Unicycler assembly failed, skip QUAST/BUSCO")

    if use_spades:
        logger.info("===== Start SPAdes Assembly =====")
        sp_fasta = spades_hybrid_assemble(
            logger=logger,
            r1_list=r1_list,
            r2_list=r2_list,
            long_read_runs=long_read_runs,
            raw_fastq_dir=DIR_RAW_FASTQ,
            clean_long_dir=DIR_CLEAN_LONG,
            skip_clean=skip_clean,
            assemble_dir=DIR_ASSEMBLY_SP,
        )
        if sp_fasta is not None:
            run_quast(logger, sp_fasta, DIR_QC / "quast_spades", est_genome_size)
            run_busco(logger, sp_fasta, DIR_QC / "busco_spades")
        else:
            logger.warning("SPAdes assembly failed, skip QUAST/BUSCO")

    if use_hybracter:
        logger.info("===== Start Hybracter Assembly =====")
        hyb_fasta = hybracter_hybrid_assemble(
            logger=logger,
            accession=accession,
            r1_list=r1_list,
            r2_list=r2_list,
            long_read_runs=long_read_runs,
            raw_fastq_dir=DIR_RAW_FASTQ,
            clean_long_dir=DIR_CLEAN_LONG,
            skip_clean=skip_clean,
            assemble_dir=DIR_ASSEMBLY_HYB,
            est_genome_size=est_genome_size
        )
        if hyb_fasta is not None:
            run_quast(logger, hyb_fasta, DIR_QC / "quast_hybracter", est_genome_size)
            run_busco(logger, hyb_fasta, DIR_QC / "busco_hybracter")
        else:
            logger.warning("Hybracter assembly failed, skip QUAST/BUSCO")

    if use_flye:
        logger.info("===== Start Flye Assembly =====")
        flye_fasta = flye_hybrid_assemble(
            logger=logger,
            long_read_runs=long_read_runs,
            raw_fastq_dir=DIR_RAW_FASTQ,
            clean_long_dir=DIR_CLEAN_LONG,
            skip_clean=skip_clean,
            assemble_dir=DIR_ASSEMBLY_FLYE,
            est_genome_size=est_genome_size
        )
        if flye_fasta is not None:
            run_quast(logger, flye_fasta, DIR_QC / "quast_flye", est_genome_size)
            run_busco(logger, flye_fasta, DIR_QC / "busco_flye")
        else:
            logger.warning("Flye assembly failed, skip QUAST/BUSCO")

    logger.info(f"========== All tasks completed for {accession} pipeline ==========")
    res_info = []
    if uni_fasta:
        res_info.append(f"Unicycler: {uni_fasta.resolve()}")
    if sp_fasta:
        res_info.append(f"SPAdes: {sp_fasta.resolve()}")
    if hyb_fasta:
        res_info.append(f"Hybracter: {hyb_fasta.resolve()}")
    if flye_fasta:
        res_info.append(f"Flye: {flye_fasta.resolve()}")
    logger.info("Assembly output paths:\n" + "\n".join(res_info))
    logger.info(f"QC report directory: {DIR_QC.resolve()}")

    return {
        "unicycler_fasta": uni_fasta,
        "spades_fasta": sp_fasta,
        "hybracter_fasta": hyb_fasta,
        "flye_fasta": flye_fasta
    }

In [2]:
import pandas as pd
import ast
import os

genus_name = 'Escherichia'

acc_bio = pd.read_csv('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/biosample_info.tsv', sep='\t')
acc_time = pd.read_csv('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/assembly_submission_info.tsv', sep='\t')
acc_time["submissionDate"] = pd.to_datetime(acc_time["submissionDate"])

target_dir = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
replicon_data = pd.read_csv(f'{target_dir}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
all_trans = replicon_data[replicon_data['category-pident_90']=='intermediate replicon'].copy()
all_trans['acc_n'] = all_trans['accession'].str.split('-').str[0]
all_acc = set(all_trans['acc_n'])

filted_bio = acc_bio[(acc_bio['accession'].isin(all_acc)) & (acc_bio['srr_list'] != '[]')]
filted_bio = pd.merge(filted_bio, acc_time, how='left', on='accession')
filted_bio = filted_bio.sort_values(by="submissionDate", ascending=False, ignore_index=True)

count = 0

for idx in filted_bio.index:
    short_read, long_read = False, False
    srr_list = ast.literal_eval(filted_bio.loc[idx, 'srr_list'])
    for run in srr_list:
        if run['platform'] == 'ILLUMINA':
            short_read = True
        if run['platform'] == 'OXFORD_NANOPORE' or run['platform'] == 'PACBIO_SMRT':
            long_read = True
    if short_read and long_read:
        print(filted_bio.loc[idx, 'organismName'], filted_bio.loc[idx, 'accession'], filted_bio.loc[idx, 'srr_list'])
        count += 1
    else:
        continue

    res = hybrid_assemble_genome(
        accession=filted_bio.loc[idx, 'accession'],
        run_info_list=srr_list,
        use_unicycler=True,
        use_spades=False,
        use_hybracter=True,
        use_flye=True,
        skip_clean=False,
        skip_qc=False
    )

    print("===== Assembly Output Paths =====")
    if res["unicycler_fasta"]:
        print(f"Unicycler fasta: {res['unicycler_fasta'].resolve()}")
    if res["spades_fasta"]:
        print(f"SPAdes scaffolds fasta: {res['spades_fasta'].resolve()}")
    if res["hybracter_fasta"]:
        print(f"Hybracter fasta: {res['hybracter_fasta'].resolve()}")
    if res["flye_fasta"]:
        print(f"flye fasta: {res['flye_fasta'].resolve()}")

2026-07-29 17:11:22,195 - INFO - ========== Hybrid assembly pipeline started for GCF_049949635.1 ==========
2026-07-29 17:11:22,196 - INFO - Working directory: /active-data/genome_re-assemble/GCF_049949635.1
2026-07-29 17:11:22,196 - INFO - Assemblers enabled: Uni=True, SPAdes=False, Hyb=True, Flye=True
2026-07-29 17:11:22,197 - INFO - skip_clean=False, skip_qc=False
2026-07-29 17:11:22,197 - INFO - Start validating dependency software versions
2026-07-29 17:11:22,198 - INFO - [Validate vdb-config] Execute command: vdb-config --version
2026-07-29 17:11:22,220 - INFO - [Validate vdb-config] Execution completed
2026-07-29 17:11:22,221 - INFO - [Validate vdb-validate] Execute command: vdb-validate --version
2026-07-29 17:11:22,241 - INFO - [Validate vdb-validate] Execution completed
2026-07-29 17:11:22,242 - INFO - [Validate prefetch] Execute command: prefetch --version
2026-07-29 17:11:22,282 - INFO - [Validate prefetch] Execution completed
2026-07-29 17:11:22,283 - INFO - [Validate fast

Escherichia coli GCF_049949635.1 [{'run_id': 'SRR32729544', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}, {'run_id': 'SRR32729613', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}, {'run_id': 'SRR36889076', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}]


2026-07-29 17:11:22,551 - INFO - [Validate fastqc] Execution completed
2026-07-29 17:11:22,553 - INFO - [Validate fastp] Execute command: fastp --version
2026-07-29 17:11:22,583 - INFO - [Validate fastp] Execution completed
2026-07-29 17:11:22,585 - INFO - [Validate quast] Execute command: quast.py --version
2026-07-29 17:11:22,794 - INFO - [Validate quast] Execution completed
2026-07-29 17:11:22,796 - INFO - [Validate busco] Execute command: busco --version
2026-07-29 17:11:23,313 - INFO - [Validate busco] Execution completed
2026-07-29 17:11:23,315 - INFO - [Validate unicycler] Execute command: unicycler --version
2026-07-29 17:11:23,403 - INFO - [Validate unicycler] Execution completed
2026-07-29 17:11:23,405 - INFO - [Validate Hybracter] Execute command: hybracter version
2026-07-29 17:11:23,490 - INFO - [Validate Hybracter] Execution completed
2026-07-29 17:11:23,491 - INFO - [Validate Flye] Execute command: flye --version
2026-07-29 17:11:23,564 - INFO - [Validate Flye] Execution

===== Assembly Output Paths =====
Unicycler fasta: /active-data/genome_re-assemble/GCF_049949635.1/assembly_unicycler/assembly.fasta
Hybracter fasta: /active-data/genome_re-assemble/GCF_049949635.1/assembly_hybracter/FINAL_OUTPUT/complete/GCF_049949635.1_final.fasta
flye fasta: /active-data/genome_re-assemble/GCF_049949635.1/assembly_flye/assembly.fasta
Escherichia coli GCF_964200005.1 [{'run_id': 'ERR13363630', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}, {'run_id': 'ERR13363302', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}]


2026-07-29 17:27:38,721 - INFO - [Validate fastqc] Execution completed
2026-07-29 17:27:38,722 - INFO - [Validate fastp] Execute command: fastp --version
2026-07-29 17:27:38,752 - INFO - [Validate fastp] Execution completed
2026-07-29 17:27:38,754 - INFO - [Validate quast] Execute command: quast.py --version
2026-07-29 17:27:38,957 - INFO - [Validate quast] Execution completed
2026-07-29 17:27:38,959 - INFO - [Validate busco] Execute command: busco --version
2026-07-29 17:27:39,444 - INFO - [Validate busco] Execution completed
2026-07-29 17:27:39,445 - INFO - [Validate unicycler] Execute command: unicycler --version
2026-07-29 17:27:39,531 - INFO - [Validate unicycler] Execution completed
2026-07-29 17:27:39,533 - INFO - [Validate Hybracter] Execute command: hybracter version
2026-07-29 17:27:39,615 - INFO - [Validate Hybracter] Execution completed
2026-07-29 17:27:39,617 - INFO - [Validate Flye] Execute command: flye --version
2026-07-29 17:27:39,692 - INFO - [Validate Flye] Execution

...

===== Assembly Output Paths =====
Unicycler fasta: /active-data/genome_re-assemble/GCF_010365325.1/assembly_unicycler/assembly.fasta
Hybracter fasta: /active-data/genome_re-assemble/GCF_010365325.1/assembly_hybracter/FINAL_OUTPUT/complete/GCF_010365325.1_final.fasta
flye fasta: /active-data/genome_re-assemble/GCF_010365325.1/assembly_flye/assembly.fasta
Escherichia coli GCF_005389605.2 [{'run_id': 'DRR102940', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}, {'run_id': 'DRR252635', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}]


2026-07-30 05:18:56,692 - INFO - [Validate fastqc] Execution completed
2026-07-30 05:18:56,693 - INFO - [Validate fastp] Execute command: fastp --version
2026-07-30 05:18:56,719 - INFO - [Validate fastp] Execution completed
2026-07-30 05:18:56,721 - INFO - [Validate quast] Execute command: quast.py --version
2026-07-30 05:18:56,910 - INFO - [Validate quast] Execution completed
2026-07-30 05:18:56,913 - INFO - [Validate busco] Execute command: busco --version
2026-07-30 05:18:57,428 - INFO - [Validate busco] Execution completed
2026-07-30 05:18:57,430 - INFO - [Validate unicycler] Execute command: unicycler --version
2026-07-30 05:18:57,512 - INFO - [Validate unicycler] Execution completed
2026-07-30 05:18:57,514 - INFO - [Validate Hybracter] Execute command: hybracter version
2026-07-30 05:18:57,590 - INFO - [Validate Hybracter] Execution completed
2026-07-30 05:18:57,592 - INFO - [Validate Flye] Execute command: flye --version
2026-07-30 05:18:57,664 - INFO - [Validate Flye] Execution

===== Assembly Output Paths =====
Unicycler fasta: /active-data/genome_re-assemble/GCF_005389605.2/assembly_unicycler/assembly.fasta
Hybracter fasta: /active-data/genome_re-assemble/GCF_005389605.2/assembly_hybracter/FINAL_OUTPUT/complete/GCF_005389605.2_final.fasta
flye fasta: /active-data/genome_re-assemble/GCF_005389605.2/assembly_flye/assembly.fasta
Escherichia coli GCF_002180135.1 [{'run_id': 'SRR4026005', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}, {'run_id': 'SRR5168522', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}, {'run_id': 'SRR5168523', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}, {'run_id': 'SRR5750477', 'platform': 'PACBIO_SMRT', 'layout': 'SINGLE'}]


2026-07-30 05:57:31,881 - INFO - [Validate fastqc] Execution completed
2026-07-30 05:57:31,882 - INFO - [Validate fastp] Execute command: fastp --version
2026-07-30 05:57:31,908 - INFO - [Validate fastp] Execution completed
2026-07-30 05:57:31,910 - INFO - [Validate quast] Execute command: quast.py --version
2026-07-30 05:57:32,109 - INFO - [Validate quast] Execution completed
2026-07-30 05:57:32,111 - INFO - [Validate busco] Execute command: busco --version
2026-07-30 05:57:32,588 - INFO - [Validate busco] Execution completed
2026-07-30 05:57:32,590 - INFO - [Validate unicycler] Execute command: unicycler --version
2026-07-30 05:57:32,673 - INFO - [Validate unicycler] Execution completed
2026-07-30 05:57:32,675 - INFO - [Validate Hybracter] Execute command: hybracter version
2026-07-30 05:57:32,756 - INFO - [Validate Hybracter] Execution completed
2026-07-30 05:57:32,758 - INFO - [Validate Flye] Execute command: flye --version
2026-07-30 05:57:32,830 - INFO - [Validate Flye] Execution

===== Assembly Output Paths =====
Unicycler fasta: /active-data/genome_re-assemble/GCF_002180135.1/assembly_unicycler/assembly.fasta
Hybracter fasta: /active-data/genome_re-assemble/GCF_002180135.1/assembly_hybracter/FINAL_OUTPUT/incomplete/GCF_002180135.1_final.fasta
flye fasta: /active-data/genome_re-assemble/GCF_002180135.1/assembly_flye/assembly.fasta
